### Alignment Results
This notebook has the objective to analize the results from the alignment between the CDS from all metagenomes and proteins associated with antibiotic biosynthesis.

### 1. Preparing the Data
Here we import the Uniprot alignment result (in this case runned in HPC-USP) and filter the best alignment hit.

In [13]:
import pandas as pd
import re

In [22]:
# Importing the results from Uniprot alignment.
df = pd.read_csv("./df_filt_all_metagenomes.csv", sep=None, engine="python")
df

,query,subject,pident,align_len,qlen,slen,evalue,bitscore,title,coverage
0,MGYG000296020_ctg64_13,tr|A0A9Q0N7D4|A0A9Q0N7D4_9DIPT,35.338,399,481,1348,4.780000e-65,231.0,tr|A0A9Q0N7D4|A0A9Q0N7D4_9DIPT Penicillin-bind...,82.952183
1,MGYG000296020_ctg1_31,tr|A0A4P6EXD3|A0A4P6EXD3_9BACL,30.488,410,413,629,1.500000e-53,191.0,tr|A0A4P6EXD3|A0A4P6EXD3_9BACL ATP-binding cas...,99.273608
2,MGYG000296020_ctg1_31,tr|A0ABT6DHE3|A0ABT6DHE3_9BACT,32.039,412,413,659,1.540000e-50,183.0,tr|A0ABT6DHE3|A0ABT6DHE3_9BACT ABC transporter...,99.757869
3,MGYG000296020_ctg1_31,tr|K7ZBJ2|K7ZBJ2_BDEBC,31.490,416,413,660,4.670000e-45,168.0,tr|K7ZBJ2|K7ZBJ2_BDEBC Macrolide ABC transport...,100.726392
4,MGYG000296020_ctg1_31,tr|A0ABU5VYP2|A0ABU5VYP2_9BACT,30.500,400,413,648,6.950000e-45,167.0,tr|A0ABU5VYP2|A0ABU5VYP2_9BACT ABC transporter...,96.852300
...,...,...,...,...,...,...,...,...,...,...
28317,MGYG000296015_ctg34_30,tr|A0A8E1W8M9|A0A8E1W8M9_9PSEU,30.556,324,430,401,3.480000e-29,120.0,tr|A0A8E1W8M9|A0A8E1W8M9_9PSEU Acetylornithine...,75.348837
28318,MGYG000296015_ctg34_30,tr|A0ABW5IAR0|A0ABW5IAR0_9PSEU,30.124,322,430,401,5.060000e-29,120.0,tr|A0ABW5IAR0|A0ABW5IAR0_9PSEU Acetylornithine...,74.883721
28319,MGYG000296015_ctg34_30,tr|A0A1I5ZLC7|A0A1I5ZLC7_9PSEU,31.348,319,430,401,8.470000e-29,119.0,tr|A0A1I5ZLC7|A0A1I5ZLC7_9PSEU Acetylornithine...,74.186047
28320,MGYG000296015_ctg34_30,tr|A0A7W3W422|A0A7W3W422_9PSEU,30.864,324,430,401,1.370000e-28,119.0,tr|A0A7W3W422|A0A7W3W422_9PSEU Acetylornithine...,75.348837


In [25]:
# Taking the BEST alignment for each BGC's CDS. We are using the smallest "evalue"
df["protein_ID"] = df["query"]

df_grouped_filtered = df.groupby("query", as_index=True).apply(lambda x: x.nsmallest(1, "evalue"))
df_grouped_filtered["protein_description"] = df_grouped_filtered["title"].str.extract(r'\|[^|]+\|[^ ]+ (.*?) OS=')
df_grouped_filtered["Organism"] = df_grouped_filtered["title"].str.extract(r'OS=(.*?) OX=')
df_grouped_filtered["TaxID"] = df_grouped_filtered["title"].str.extract(r'OX=(\d+)')
df_grouped_filtered["Gene"] = df_grouped_filtered["title"].str.extract(r'GN=([^ ]+)')
df_grouped_filtered["Evidence"] = df_grouped_filtered["title"].str.extract(r'PE=(\d+)')
df_grouped_filtered["Version"] = df_grouped_filtered["title"].str.extract(r'SV=(\d+)')
df_grouped_filtered = df_grouped_filtered.drop(columns="title")

df_grouped_filtered

,,subject,pident,align_len,qlen,slen,evalue,bitscore,coverage,protein_ID,protein_description,Organism,TaxID,Gene,Evidence,Version
query,,,,,,,,,,,,,,,,
MGYG000296008_ctg22_13,24551,tr|A0A6M1PI93|A0A6M1PI93_9BACL,30.108,186,257,2026,3.740000e-19,90.1,72.373541,MGYG000296008_ctg22_13,Amino acid adenylation domain-containing protein,Paenibacillus apii,1850370,G5B47_10895,3,1
MGYG000296008_ctg22_15,24557,tr|A0A1V0U7E9|A0A1V0U7E9_STRVN,53.846,260,260,260,2.640000e-89,268.0,100.000000,MGYG000296008_ctg22_15,Short chain dehydrogenase,Streptomyces violaceoruber,1935,B1H20_06975,3,1
MGYG000296008_ctg22_2,24522,tr|A0A4U0S3Y9|A0A4U0S3Y9_9ACTN,30.550,527,505,4657,7.980000e-65,231.0,104.356436,MGYG000296008_ctg22_2,SDR family NAD(P)-dependent oxidoreductase,Actinacidiphila oryziradicis,2571141,FCI23_38020,4,1
MGYG000296008_ctg22_6,24550,sp|P95483|PRND_PSEFL,32.727,110,143,363,1.180000e-14,72.4,76.923077,MGYG000296008_ctg22_6,Aminopyrrolnitrin oxygenase PrnD,Pseudomonas fluorescens,294,prnD,1,1
MGYG000296008_ctg2_15,24583,tr|A0A1Q2HL76|A0A1Q2HL76_9BACT,38.732,142,140,459,2.390000e-16,77.4,101.428571,MGYG000296008_ctg2_15,Undecaprenyl-diphosphatase,Sedimentisphaera cyanobacteriorum,1940790,uppP,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MGYG000296076_ctg1_130,23230,tr|A0A6N7Z877|A0A6N7Z877_9PSEU,43.363,339,338,353,2.900000e-83,259.0,100.295858,MGYG000296076_ctg1_130,Aminotransferase class I/II-fold pyridoxal pho...,Amycolatopsis pithecellobii,664692,GKO32_27080,3,1
MGYG000296076_ctg1_133,23323,sp|Q9L9F8|NOVJ_STRNV,39.754,244,248,262,1.400000e-48,164.0,98.387097,MGYG000296076_ctg1_133,Short-chain reductase protein NovJ,Streptomyces niveus,193462,novJ,1,1
MGYG000296076_ctg1_135,23566,tr|A0A0U5JEC7|A0A0U5JEC7_9BACT,32.794,247,265,511,2.450000e-35,135.0,93.207547,MGYG000296076_ctg1_135,Beta-lactamase,Candidatus Protochlamydia naegleriophila,389348,PNK_1118,3,1


### 2. Making columns with usefull information
Above we can see the best results for each hit in alignment (The best results were choose by the lowest evalue presented). Here we need to create columns with the following informations: total hits for each BGC, all protein descriptions (in order), all organisms (in order), all genes (in order) and all taxID (in order).

In [26]:
# Creating the column for BGC IDs
df_grouped_filtered["BGC_ID"] = (
    df_grouped_filtered["protein_ID"]
    .str.extract(r'^(MGYG\d+)_ctg(\d+)')
    .agg("_".join, axis=1)
)
df_grouped_filtered

,,subject,pident,align_len,qlen,slen,evalue,bitscore,coverage,protein_ID,protein_description,Organism,TaxID,Gene,Evidence,Version,BGC_ID
query,,,,,,,,,,,,,,,,,
MGYG000296008_ctg22_13,24551,tr|A0A6M1PI93|A0A6M1PI93_9BACL,30.108,186,257,2026,3.740000e-19,90.1,72.373541,MGYG000296008_ctg22_13,Amino acid adenylation domain-containing protein,Paenibacillus apii,1850370,G5B47_10895,3,1,MGYG000296008_22
MGYG000296008_ctg22_15,24557,tr|A0A1V0U7E9|A0A1V0U7E9_STRVN,53.846,260,260,260,2.640000e-89,268.0,100.000000,MGYG000296008_ctg22_15,Short chain dehydrogenase,Streptomyces violaceoruber,1935,B1H20_06975,3,1,MGYG000296008_22
MGYG000296008_ctg22_2,24522,tr|A0A4U0S3Y9|A0A4U0S3Y9_9ACTN,30.550,527,505,4657,7.980000e-65,231.0,104.356436,MGYG000296008_ctg22_2,SDR family NAD(P)-dependent oxidoreductase,Actinacidiphila oryziradicis,2571141,FCI23_38020,4,1,MGYG000296008_22
MGYG000296008_ctg22_6,24550,sp|P95483|PRND_PSEFL,32.727,110,143,363,1.180000e-14,72.4,76.923077,MGYG000296008_ctg22_6,Aminopyrrolnitrin oxygenase PrnD,Pseudomonas fluorescens,294,prnD,1,1,MGYG000296008_22
MGYG000296008_ctg2_15,24583,tr|A0A1Q2HL76|A0A1Q2HL76_9BACT,38.732,142,140,459,2.390000e-16,77.4,101.428571,MGYG000296008_ctg2_15,Undecaprenyl-diphosphatase,Sedimentisphaera cyanobacteriorum,1940790,uppP,3,1,MGYG000296008_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MGYG000296076_ctg1_130,23230,tr|A0A6N7Z877|A0A6N7Z877_9PSEU,43.363,339,338,353,2.900000e-83,259.0,100.295858,MGYG000296076_ctg1_130,Aminotransferase class I/II-fold pyridoxal pho...,Amycolatopsis pithecellobii,664692,GKO32_27080,3,1,MGYG000296076_1
MGYG000296076_ctg1_133,23323,sp|Q9L9F8|NOVJ_STRNV,39.754,244,248,262,1.400000e-48,164.0,98.387097,MGYG000296076_ctg1_133,Short-chain reductase protein NovJ,Streptomyces niveus,193462,novJ,1,1,MGYG000296076_1
MGYG000296076_ctg1_135,23566,tr|A0A0U5JEC7|A0A0U5JEC7_9BACT,32.794,247,265,511,2.450000e-35,135.0,93.207547,MGYG000296076_ctg1_135,Beta-lactamase,Candidatus Protochlamydia naegleriophila,389348,PNK_1118,3,1,MGYG000296076_1


In [27]:
# Creating the column with the number of hits for each BGC CDS.
df_grouped_filtered["number_of_uniprot_hits"] = df_grouped_filtered.groupby("BGC_ID")["protein_ID"].transform("count")
df_grouped_filtered

,,subject,pident,align_len,qlen,slen,evalue,bitscore,coverage,protein_ID,protein_description,Organism,TaxID,Gene,Evidence,Version,BGC_ID,number_of_uniprot_hits
query,,,,,,,,,,,,,,,,,,
MGYG000296008_ctg22_13,24551,tr|A0A6M1PI93|A0A6M1PI93_9BACL,30.108,186,257,2026,3.740000e-19,90.1,72.373541,MGYG000296008_ctg22_13,Amino acid adenylation domain-containing protein,Paenibacillus apii,1850370,G5B47_10895,3,1,MGYG000296008_22,4
MGYG000296008_ctg22_15,24557,tr|A0A1V0U7E9|A0A1V0U7E9_STRVN,53.846,260,260,260,2.640000e-89,268.0,100.000000,MGYG000296008_ctg22_15,Short chain dehydrogenase,Streptomyces violaceoruber,1935,B1H20_06975,3,1,MGYG000296008_22,4
MGYG000296008_ctg22_2,24522,tr|A0A4U0S3Y9|A0A4U0S3Y9_9ACTN,30.550,527,505,4657,7.980000e-65,231.0,104.356436,MGYG000296008_ctg22_2,SDR family NAD(P)-dependent oxidoreductase,Actinacidiphila oryziradicis,2571141,FCI23_38020,4,1,MGYG000296008_22,4
MGYG000296008_ctg22_6,24550,sp|P95483|PRND_PSEFL,32.727,110,143,363,1.180000e-14,72.4,76.923077,MGYG000296008_ctg22_6,Aminopyrrolnitrin oxygenase PrnD,Pseudomonas fluorescens,294,prnD,1,1,MGYG000296008_22,4
MGYG000296008_ctg2_15,24583,tr|A0A1Q2HL76|A0A1Q2HL76_9BACT,38.732,142,140,459,2.390000e-16,77.4,101.428571,MGYG000296008_ctg2_15,Undecaprenyl-diphosphatase,Sedimentisphaera cyanobacteriorum,1940790,uppP,3,1,MGYG000296008_2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MGYG000296076_ctg1_130,23230,tr|A0A6N7Z877|A0A6N7Z877_9PSEU,43.363,339,338,353,2.900000e-83,259.0,100.295858,MGYG000296076_ctg1_130,Aminotransferase class I/II-fold pyridoxal pho...,Amycolatopsis pithecellobii,664692,GKO32_27080,3,1,MGYG000296076_1,3
MGYG000296076_ctg1_133,23323,sp|Q9L9F8|NOVJ_STRNV,39.754,244,248,262,1.400000e-48,164.0,98.387097,MGYG000296076_ctg1_133,Short-chain reductase protein NovJ,Streptomyces niveus,193462,novJ,1,1,MGYG000296076_1,3
MGYG000296076_ctg1_135,23566,tr|A0A0U5JEC7|A0A0U5JEC7_9BACT,32.794,247,265,511,2.450000e-35,135.0,93.207547,MGYG000296076_ctg1_135,Beta-lactamase,Candidatus Protochlamydia naegleriophila,389348,PNK_1118,3,1,MGYG000296076_1,3


In [28]:
# Making columns with proteins ID, description, organism, TaxID and gene for each BGC
def safe_join(series):
    return ", ".join(series.fillna("").astype(str))
df_grouped_filtered_with_bgc = (
    df_grouped_filtered.groupby("BGC_ID")
      .apply(lambda x: pd.Series({
          "UniProt_alignment_hits": safe_join(x["protein_ID"]),
          "UniProt_protein_descriptions": safe_join(x["protein_description"]),
          "UniProt_protein_organism": safe_join(x["Organism"]),
          "UniProt_protein_TaxID": safe_join(x["TaxID"]),
          "UniProt_protein_gene": safe_join(x["Gene"])
      }))
      .reset_index()
)
df_grouped_filtered_with_bgc

,BGC_ID,UniProt_alignment_hits,UniProt_protein_descriptions,UniProt_protein_organism,UniProt_protein_TaxID,UniProt_protein_gene
0,MGYG000296008_2,MGYG000296008_ctg2_15,Undecaprenyl-diphosphatase,Sedimentisphaera cyanobacteriorum,1940790,uppP
1,MGYG000296008_22,"MGYG000296008_ctg22_13, MGYG000296008_ctg22_15...",Amino acid adenylation domain-containing prote...,"Paenibacillus apii, Streptomyces violaceoruber...","1850370, 1935, 2571141, 294","G5B47_10895, B1H20_06975, FCI23_38020, prnD"
2,MGYG000296008_6,"MGYG000296008_ctg6_39, MGYG000296008_ctg6_44, ...","Small ribosomal subunit protein uS4, Enoyl-CoA...","Escherichia coli (strain K12), Amycolatopsis o...","83333, 31958, 224308","rpsD, dpgD, pksF"
3,MGYG000296009_1,"MGYG000296009_ctg1_80, MGYG000296009_ctg1_83","Ketoreductase, Beta-ketoacyl-[acyl-carrier-pro...","Streptomyces violaceoruber, Streptomyces coeli...","1935, 100226","gra-orf6, redP"
4,MGYG000296014_17,MGYG000296014_ctg17_41,Probable polyketide biosynthesis zinc-dependen...,Bacillus subtilis (strain 168),224308,pksB
...,...,...,...,...,...,...
66,MGYG000296075_20,MGYG000296075_ctg20_26,cysteine-S-conjugate beta-lyase,Amycolatopsis taiwanensis,342230,Atai01_46660
67,MGYG000296075_4,MGYG000296075_ctg4_24,DNA-binding response regulator,Enterococcus malodoratus ATCC 43197,1158601,I585_02216
68,MGYG000296076_1,"MGYG000296076_ctg1_130, MGYG000296076_ctg1_133...",Aminotransferase class I/II-fold pyridoxal pho...,"Amycolatopsis pithecellobii, Streptomyces nive...","664692, 193462, 389348","GKO32_27080, novJ, PNK_1118"
69,MGYG000296076_10,"MGYG000296076_ctg10_30, MGYG000296076_ctg10_37","SDR family oxidoreductase, Lysine--tRNA ligase","Streptomyces rubrogriseus, Capillimicrobium pa...","194673, 2884022","G3I66_28110, epmA"


### 3. Uniting the Alignment information with BGCs information
In this section we merge the dataframe of the alignment with the results from antiSMASH, BIG-SCAPE, POEM and DeepSEA. They are in /home/pedro/resultados_bgc/fna_dos_bgcs/updated_BGCs_information.csv table.

In [29]:
updated_bgc_table = pd.read_csv("/home/pedro/resultados_bgc/fna_dos_bgcs/updated_BGCs_information.csv")
updated_bgc_table = updated_bgc_table.rename(columns={"metagenome_id": "BGC_ID"})
updated_bgc_table

,Unnamed: 0,BGC_ID,Reference_BGC,distance,jaccard,adjacency,dss,reference_BGC_class,reference_compound_name,has_resistance_protein,resistance_protein_class,resistance_protein_count,antismash_annotation,num_proteins_antismash_annotation,strand,coordinates
0,0,MGYG000296006_405,BGC0002398,0.47,1.00,1.00,0.38,terpene,(-)-ent-quiannulatene,False,NaN,NaN,NaN,NaN,NaN,NaN
1,1,MGYG000296006_71,BGC0002132,0.74,0.67,0.50,0.14,ribosomal,group 1 methanobactin,False,NaN,NaN,NaN,NaN,+,ctg71_3 --> ctg71_4 --> ctg71_5 --> ctg71_6 --...
2,2,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,NaN,NaN,NaN,NaN,+,ctg2_6 --> ctg2_7 --> ctg2_8 --> ctg2_9 --> ct...
3,3,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,NaN,NaN,NaN,NaN,-,ctg2_2 <-- ctg2_3 <-- ctg2_4 <-- ctg2_5 / ctg2...
4,4,MGYG000296008_22,BGC0000924,0.81,0.50,0.33,0.10,other,pyrrolnitrin,False,NaN,NaN,NaN,NaN,-,ctg22_2 <-- ctg22_3 <-- ctg22_4 <-- ctg22_5 <-...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,165,MGYG000296076_1,BGC0000656,0.91,0.33,0.00,0.04,terpene,zeaxanthin,False,NaN,NaN,NaN,NaN,-,ctg1_139 <-- ctg1_140 <-- ctg1_141 / ctg1_148 ...
166,166,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,NaN,NaN,carotenoid,3.0,+,3/548 --> ctg10_23 --> ctg10_24 --> ctg10_25
167,167,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,NaN,NaN,carotenoid,3.0,-,ctg10_29 <-- ctg10_30 <-- ctg10_31 <-- ctg10_3...
168,168,MGYG000296076_4,BGC0002865,0.56,1.00,1.00,0.26,PKS,Aloesone,False,NaN,NaN,NaN,NaN,+,ctg4_63 --> ctg4_64 --> ctg4_65 --> ctg4_66 / ...


In [30]:
merge_bgc_table =pd.merge(updated_bgc_table, df_grouped_filtered_with_bgc, on="BGC_ID", how="left")
merge_bgc_table

,Unnamed: 0,BGC_ID,Reference_BGC,distance,jaccard,adjacency,dss,reference_BGC_class,reference_compound_name,has_resistance_protein,...,resistance_protein_count,antismash_annotation,num_proteins_antismash_annotation,strand,coordinates,UniProt_alignment_hits,UniProt_protein_descriptions,UniProt_protein_organism,UniProt_protein_TaxID,UniProt_protein_gene
0,0,MGYG000296006_405,BGC0002398,0.47,1.00,1.00,0.38,terpene,(-)-ent-quiannulatene,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,MGYG000296006_71,BGC0002132,0.74,0.67,0.50,0.14,ribosomal,group 1 methanobactin,False,...,NaN,NaN,NaN,+,ctg71_3 --> ctg71_4 --> ctg71_5 --> ctg71_6 --...,NaN,NaN,NaN,NaN,NaN
2,2,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,...,NaN,NaN,NaN,+,ctg2_6 --> ctg2_7 --> ctg2_8 --> ctg2_9 --> ct...,MGYG000296008_ctg2_15,Undecaprenyl-diphosphatase,Sedimentisphaera cyanobacteriorum,1940790,uppP
3,3,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,...,NaN,NaN,NaN,-,ctg2_2 <-- ctg2_3 <-- ctg2_4 <-- ctg2_5 / ctg2...,MGYG000296008_ctg2_15,Undecaprenyl-diphosphatase,Sedimentisphaera cyanobacteriorum,1940790,uppP
4,4,MGYG000296008_22,BGC0000924,0.81,0.50,0.33,0.10,other,pyrrolnitrin,False,...,NaN,NaN,NaN,-,ctg22_2 <-- ctg22_3 <-- ctg22_4 <-- ctg22_5 <-...,"MGYG000296008_ctg22_13, MGYG000296008_ctg22_15...",Amino acid adenylation domain-containing prote...,"Paenibacillus apii, Streptomyces violaceoruber...","1850370, 1935, 2571141, 294","G5B47_10895, B1H20_06975, FCI23_38020, prnD"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,165,MGYG000296076_1,BGC0000656,0.91,0.33,0.00,0.04,terpene,zeaxanthin,False,...,NaN,NaN,NaN,-,ctg1_139 <-- ctg1_140 <-- ctg1_141 / ctg1_148 ...,"MGYG000296076_ctg1_130, MGYG000296076_ctg1_133...",Aminotransferase class I/II-fold pyridoxal pho...,"Amycolatopsis pithecellobii, Streptomyces nive...","664692, 193462, 389348","GKO32_27080, novJ, PNK_1118"
166,166,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,...,NaN,carotenoid,3.0,+,3/548 --> ctg10_23 --> ctg10_24 --> ctg10_25,"MGYG000296076_ctg10_30, MGYG000296076_ctg10_37","SDR family oxidoreductase, Lysine--tRNA ligase","Streptomyces rubrogriseus, Capillimicrobium pa...","194673, 2884022","G3I66_28110, epmA"
167,167,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,...,NaN,carotenoid,3.0,-,ctg10_29 <-- ctg10_30 <-- ctg10_31 <-- ctg10_3...,"MGYG000296076_ctg10_30, MGYG000296076_ctg10_37","SDR family oxidoreductase, Lysine--tRNA ligase","Streptomyces rubrogriseus, Capillimicrobium pa...","194673, 2884022","G3I66_28110, epmA"
168,168,MGYG000296076_4,BGC0002865,0.56,1.00,1.00,0.26,PKS,Aloesone,False,...,NaN,NaN,NaN,+,ctg4_63 --> ctg4_64 --> ctg4_65 --> ctg4_66 / ...,"MGYG000296076_ctg4_64, MGYG000296076_ctg4_71","Lipid A 4'-phosphatase, ABC transporter family...",Porphyromonas gingivalis (strain ATCC 33277 / ...,"431947, 46176","lpxF, EDD27_3505"


In [31]:
merge_bgc_table.to_csv("bgc_table_with_alignment_information.csv", index=False)